# Práctica de Python: Minería de Texto
**Entorno:** Google Colab · **Lenguaje:** Python

Este cuaderno realiza limpieza de texto, tokenización, conteo de vocabulario, eliminación de palabras vacías y frecuencias de términos. Al final incluye retos opcionales con NLTK y stemming.

## 1. Preparación del corpus
Usaremos tres comentarios de ejemplo sobre el servicio al cliente.

In [ ]:
corpus = [
    "El servicio al cliente es excelente y muy rápido en la entrega",
    "El servicio es pésimo y la atención al cliente es muy lenta",
    "Excelente atención pero el servicio de entrega es lento"
]

for i, comentario in enumerate(corpus, start=1):
    print(f"Comentario {i}: {comentario}")

## 2. Limpieza del texto
Se eliminan URLs, correos, etiquetas HTML, emojis y signos de puntuación; también se normalizan espacios y mayúsculas.

In [ ]:
import re

def limpiar_texto(texto):
    texto = re.sub(r"https?://\\S+|www\\.\\S+", " ", texto)
    texto = re.sub(r"[\\w.+-]+@[\\w-]+(?:\\.[\\w-]+)+", " ", texto)
    texto = re.sub(r"<[^>]+>", " ", texto)
    texto = re.sub(r"[^\\w\\sáéíóúüñÁÉÍÓÚÜÑ]", " ", texto, flags=re.UNICODE)
    texto = re.sub(r"\\s+", " ", texto).strip().lower()
    return texto

corpus_limpio = [limpiar_texto(texto) for texto in corpus]
for texto in corpus_limpio:
    print(texto)

## 3. Tokenización y métricas básicas
N es el total de tokens y V es el conjunto de palabras únicas del corpus.

In [ ]:
tokens_por_documento = [texto.split() for texto in corpus_limpio]
tokens = [token for documento in tokens_por_documento for token in documento]
N = len(tokens)
V = set(tokens)

print("Total de tokens (N):", N)
print("Tamaño del vocabulario (|V|):", len(V))
print("Vocabulario:", sorted(V))

## 4. Eliminación de stopwords
Usamos el conjunto indicado en la práctica: `al`, `de`, `el`, `en`, `es`, `la`, `muy`, `pero`, `y`.

In [ ]:
stopwords = {"al", "de", "el", "en", "es", "la", "muy", "pero", "y"}
tokens_filtrados_por_documento = [
    [token for token in documento if token not in stopwords]
    for documento in tokens_por_documento
]
tokens_filtrados = [token for documento in tokens_filtrados_por_documento for token in documento]
V_filtrado = set(tokens_filtrados)
proporcion_stopwords = (N - len(tokens_filtrados)) / N

print("Tokens después de filtrar:", len(tokens_filtrados))
print("Vocabulario filtrado:", sorted(V_filtrado))
print("Proporción de stopwords eliminadas:", round(proporcion_stopwords, 2))

## 5. Frecuencia de términos
Se muestran las frecuencias globales y las frecuencias por comentario.

In [ ]:
from collections import Counter
import pandas as pd

frecuencias = Counter(tokens_filtrados)
tabla_frecuencias = pd.DataFrame(frecuencias.most_common(), columns=["Término", "Frecuencia"])
display(tabla_frecuencias)

for i, documento in enumerate(tokens_filtrados_por_documento, start=1):
    print(f"\\nComentario {i}:", dict(Counter(documento)))

## 6. Visualización
Gráfica de las palabras más frecuentes después de quitar stopwords.

In [ ]:
import matplotlib.pyplot as plt

top = frecuencias.most_common(10)
if top:
    palabras, cantidades = zip(*top)
    plt.figure(figsize=(10, 5))
    plt.bar(palabras, cantidades)
    plt.title("Términos más frecuentes (sin stopwords)")
    plt.xlabel("Término")
    plt.ylabel("Frecuencia")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Validación con aserciones
Las aserciones permiten comprobar automáticamente que las métricas coinciden con el corpus usado.

In [ ]:
assert N == 33, f"Se esperaban 33 tokens, se obtuvieron {N}"
assert len(V) == 18, f"Se esperaban 18 palabras únicas, se obtuvieron {len(V)}"
assert len(stopwords) == 9
assert len(V_filtrado) == 9, f"Se esperaban 9 términos únicos filtrados, se obtuvieron {len(V_filtrado)}"
assert round(proporcion_stopwords, 2) == 0.50
print("✅ Todas las validaciones se cumplieron.")

## 8. Reto opcional: agregar un cuarto comentario
Puedes agregar un comentario propio y volver a ejecutar las celdas desde la sección de limpieza para comparar los resultados.

In [ ]:
comentario_nuevo = "La atención fue buena, pero la entrega tardó demasiado"
corpus_con_extra = corpus + [comentario_nuevo]
print("Nuevo comentario:", comentario_nuevo)
print("Número de comentarios:", len(corpus_con_extra))
print("Para recalcular todas las métricas, asigna corpus = corpus_con_extra y vuelve a ejecutar desde la sección 2.")

## 9. Reto opcional: stopwords en español con NLTK
Esta sección descarga el recurso de NLTK en el entorno de Colab y compara el conjunto de stopwords con el conjunto pequeño de la práctica.

In [ ]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords as nltk_stopwords

stopwords_es = set(nltk_stopwords.words("spanish"))
tokens_nltk = [token for token in tokens if token not in stopwords_es]
print("Stopwords en español (NLTK):", len(stopwords_es))
print("Tokens restantes con NLTK:", len(tokens_nltk))
print("Primeras stopwords:", sorted(stopwords_es)[:20])

## 10. Reto opcional: stemming en español
El stemming reduce palabras a una raíz aproximada; la raíz no siempre es una palabra válida del español.

In [ ]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("spanish")
pares_stemming = [(palabra, stemmer.stem(palabra)) for palabra in sorted(V_filtrado)]
pd.DataFrame(pares_stemming, columns=["Palabra", "Raíz (stemming)"])

## Conclusión
La minería de texto permite convertir comentarios escritos en datos analizables. La limpieza y eliminación de stopwords facilitan identificar los términos más frecuentes, aunque el resultado depende del corpus y de las reglas de preprocesamiento.